# 🚦 Canonical Queueing Notebook — Core Formulas + DES Validation

This notebook is the canonical entry point for the **queueing** part of the course.

It covers:
- Core formulas (Little’s Law, utilization, stability)
- Closed-form M/M/1 and M/M/c (Erlang C) metrics
- A small discrete-event simulation (DES) of an FCFS queue
- Validation: simulation vs analytics
- Replication-based confidence intervals

Design goals:
- Run top-to-bottom on a fresh kernel
- Keep everything deterministic and reproducible

In [1]:
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Iterable

import numpy as np

## Core definitions

We’ll use standard notation:
- $\lambda$: arrival rate (jobs / time)
- $\mu$: service rate (jobs / time), mean service time $1/\mu$
- $c$: number of identical servers
- $\rho$: utilization
  - M/M/1: $\rho=\lambda/\mu$
  - M/M/c: $\rho=\lambda/(c\mu)$
- Stability: system is stable if $\rho<1$
- Little’s Law (steady state): $L = \lambda W$ and $L_q = \lambda W_q$

Where:
- $L$ is mean number in system, $L_q$ in queue
- $W$ is mean time in system, $W_q$ waiting time in queue

In [2]:
@dataclass(frozen=True)
class MMCMetrics:
    rho: float
    p_wait: float
    lq: float
    wq: float
    w: float
    l: float


def mm1_metrics(lam: float, mu: float) -> tuple[float, float, float, float, float]:
    """Return (rho, Lq, Wq, W, L) for an M/M/1 queue."""
    if lam <= 0 or mu <= 0:
        raise ValueError("lam and mu must be > 0")
    rho = lam / mu
    if rho >= 1.0:
        raise ValueError(f"Unstable: rho={rho:.3f} must be < 1")
    lq = (rho**2) / (1 - rho)
    wq = lq / lam
    w = wq + 1 / mu
    l = lam * w
    return rho, lq, wq, w, l


def _poisson_prob_mass(k: int, x: float) -> float:
    return math.exp(-x) * (x**k) / math.factorial(k)


def mmc_metrics(lam: float, mu: float, c: int) -> MMCMetrics:
    """Closed-form M/M/c (Erlang C) metrics."""
    if lam <= 0 or mu <= 0:
        raise ValueError("lam and mu must be > 0")
    if c <= 0:
        raise ValueError("c must be >= 1")
    a = lam / mu  # offered load
    rho = lam / (c * mu)
    if rho >= 1.0:
        raise ValueError(f"Unstable: rho={rho:.3f} must be < 1")
    # Compute P0 via standard normalization
    sum_terms = sum((a**n) / math.factorial(n) for n in range(c))
    last = (a**c) / (math.factorial(c) * (1 - rho))
    p0 = 1.0 / (sum_terms + last)
    p_wait = last * p0  # Erlang C
    lq = p_wait * (rho / (1 - rho))
    wq = lq / lam
    w = wq + 1 / mu
    l = lam * w
    return MMCMetrics(rho=rho, p_wait=p_wait, lq=lq, wq=wq, w=w, l=l)

In [3]:
# Simple numeric example
lam = 0.9
mu = 1.0
rho, lq, wq, w, l = mm1_metrics(lam=lam, mu=mu)
rho, lq, wq, w, l

(0.9,
 8.100000000000003,
 9.000000000000004,
 10.000000000000004,
 9.000000000000004)

## Discrete-event simulation (DES): FCFS M/M/1

We’ll simulate an M/M/1 queue with:
- Exponential inter-arrival times with rate $\lambda$
- Exponential service times with rate $\mu$
- FCFS discipline, single server
- Outputs: per-customer waiting time $W_q$ and sojourn time $W$

We’ll also add a warmup and then estimate means + confidence intervals across replications.

In [4]:
@dataclass(frozen=True)
class MM1SimResult:
    wq: np.ndarray  # waiting times after warmup
    w: np.ndarray   # sojourn times after warmup


def simulate_mm1_fcfs(
    lam: float,
    mu: float,
    n_customers: int,
    warmup: int,
    rng: np.random.Generator,
) -> MM1SimResult:
    if lam <= 0 or mu <= 0:
        raise ValueError("lam and mu must be > 0")
    if n_customers <= 0:
        raise ValueError("n_customers must be > 0")
    if warmup < 0 or warmup >= n_customers:
        raise ValueError("warmup must be in [0, n_customers-1]")
    # Arrival process
    inter_arrivals = rng.exponential(scale=1 / lam, size=n_customers)
    arrival_times = np.cumsum(inter_arrivals)
    # Service times
    service_times = rng.exponential(scale=1 / mu, size=n_customers)
    # Recurrence for FCFS single server
    start_service = np.empty(n_customers, dtype=float)
    departure_times = np.empty(n_customers, dtype=float)
    start_service[0] = arrival_times[0]
    departure_times[0] = start_service[0] + service_times[0]
    for i in range(1, n_customers):
        start_service[i] = max(arrival_times[i], departure_times[i - 1])
        departure_times[i] = start_service[i] + service_times[i]
    wq = start_service - arrival_times
    w = departure_times - arrival_times
    return MM1SimResult(wq=wq[warmup:], w=w[warmup:])


def mean_ci_95(x: np.ndarray) -> tuple[float, float, float]:
    """(mean, lo, hi) using normal approx on sample mean."""
    x = np.asarray(x, dtype=float)
    if x.size < 2:
        raise ValueError("Need at least 2 samples")
    m = float(x.mean())
    s = float(x.std(ddof=1))
    se = s / math.sqrt(x.size)
    z = 1.96
    return m, m - z * se, m + z * se

In [5]:
# Compare analytics vs simulation for M/M/1
lam = 0.9
mu = 1.0
rho, lq, wq_analytic, w_analytic, l = mm1_metrics(lam=lam, mu=mu)

n_replications = 50
n_customers = 50_000
warmup = 5_000

wq_means = []
w_means = []
for rep in range(n_replications):
    rng = np.random.default_rng(12345 + rep)
    sim = simulate_mm1_fcfs(lam=lam, mu=mu, n_customers=n_customers, warmup=warmup, rng=rng)
    wq_means.append(sim.wq.mean())
    w_means.append(sim.w.mean())

wq_mean, wq_lo, wq_hi = mean_ci_95(np.array(wq_means))
w_mean, w_lo, w_hi = mean_ci_95(np.array(w_means))

{
    "rho": rho,
    "Wq_analytic": wq_analytic,
    "Wq_sim_mean": wq_mean,
    "Wq_sim_95CI": (wq_lo, wq_hi),
    "W_analytic": w_analytic,
    "W_sim_mean": w_mean,
    "W_sim_95CI": (w_lo, w_hi),
}

{'rho': 0.9,
 'Wq_analytic': 9.000000000000004,
 'Wq_sim_mean': 9.013504203861157,
 'Wq_sim_95CI': (8.691033748710733, 9.33597465901158),
 'W_analytic': 10.000000000000004,
 'W_sim_mean': 10.012788535258926,
 'W_sim_95CI': (9.689406061117642, 10.336171009400209)}

## Strict CRN demo (common random numbers)

In this course repo, “strict CRN” means: when you compare alternatives (policies, system designs), you re-use the *same underlying randomness* so differences are less noisy.

Here we demonstrate CRN in a simple way: compare two M/M/1 systems with different $\mu$ using shared uniforms for inter-arrivals and services.

We’ll estimate $\Delta W = W(\mu_2) - W(\mu_1)$ in two ways:
- **Independent RNGs** per system
- **CRN** (shared uniforms, transformed to exponentials)
and compare the variance of the estimator.

In [6]:
def exp_from_uniform(u: np.ndarray, rate: float) -> np.ndarray:
    """Transform U~Unif(0,1) to Exp(rate) using inverse CDF."""
    u = np.asarray(u, dtype=float)
    if np.any((u <= 0) | (u >= 1)):
        raise ValueError("u must be in (0,1)")
    if rate <= 0:
        raise ValueError("rate must be > 0")
    return -np.log(u) / rate


def simulate_mm1_from_streams(
    inter_arrivals: np.ndarray,
    service_times: np.ndarray,
    warmup: int,
    rng: np.random.Generator | None = None,  # kept for signature symmetry; unused
 ) -> MM1SimResult:
    n_customers = int(len(inter_arrivals))
    if len(service_times) != n_customers:
        raise ValueError("inter_arrivals and service_times must have same length")
    if warmup < 0 or warmup >= n_customers:
        raise ValueError("warmup must be in [0, n_customers-1]")
    arrival_times = np.cumsum(inter_arrivals)
    start_service = np.empty(n_customers, dtype=float)
    departure_times = np.empty(n_customers, dtype=float)
    start_service[0] = arrival_times[0]
    departure_times[0] = start_service[0] + service_times[0]
    for i in range(1, n_customers):
        start_service[i] = max(arrival_times[i], departure_times[i - 1])
        departure_times[i] = start_service[i] + service_times[i]
    wq = start_service - arrival_times
    w = departure_times - arrival_times
    return MM1SimResult(wq=wq[warmup:], w=w[warmup:])


def compare_two_mmus_delta_w(
    lam: float,
    mu1: float,
    mu2: float,
    n_customers: int,
    warmup: int,
    n_replications: int,
    base_seed: int,
 ) -> dict[str, float]:
    # Independent streams
    delta_w_ind = []
    for rep in range(n_replications):
        rng1 = np.random.default_rng(base_seed + 10_000 * rep + 1)
        rng2 = np.random.default_rng(base_seed + 10_000 * rep + 2)
        s1 = simulate_mm1_fcfs(lam=lam, mu=mu1, n_customers=n_customers, warmup=warmup, rng=rng1).w.mean()
        s2 = simulate_mm1_fcfs(lam=lam, mu=mu2, n_customers=n_customers, warmup=warmup, rng=rng2).w.mean()
        delta_w_ind.append(s2 - s1)
    delta_w_ind = np.array(delta_w_ind)

    # Strict CRN: shared U's, transformed for each mu
    delta_w_crn = []
    for rep in range(n_replications):
        rng = np.random.default_rng(base_seed + 10_000 * rep + 3)
        u_arr = rng.random(n_customers)
        u_srv = rng.random(n_customers)
        inter_arrivals = exp_from_uniform(u_arr, rate=lam)
        service_1 = exp_from_uniform(u_srv, rate=mu1)
        service_2 = exp_from_uniform(u_srv, rate=mu2)
        s1 = simulate_mm1_from_streams(inter_arrivals, service_1, warmup=warmup).w.mean()
        s2 = simulate_mm1_from_streams(inter_arrivals, service_2, warmup=warmup).w.mean()
        delta_w_crn.append(s2 - s1)
    delta_w_crn = np.array(delta_w_crn)

    return {
        "deltaW_ind_mean": float(delta_w_ind.mean()),
        "deltaW_ind_sd": float(delta_w_ind.std(ddof=1)),
        "deltaW_crn_mean": float(delta_w_crn.mean()),
        "deltaW_crn_sd": float(delta_w_crn.std(ddof=1)),
        "variance_reduction_factor": float((delta_w_ind.var(ddof=1)) / (delta_w_crn.var(ddof=1))),
    }

In [7]:
lam = 0.9
mu1 = 1.0
mu2 = 1.1  # slightly faster server

compare_two_mmus_delta_w(
    lam=lam,
    mu1=mu1,
    mu2=mu2,
    n_customers=30_000,
    warmup=3_000,
    n_replications=40,
    base_seed=202601,
)

{'deltaW_ind_mean': -4.9505003958508285,
 'deltaW_ind_sd': 1.2173660222592892,
 'deltaW_crn_mean': -4.853904980177748,
 'deltaW_crn_sd': 0.8841036888817213,
 'variance_reduction_factor': 1.8959895826161937}